In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Better looking plots
plt.style.use("ggplot")

In [3]:
df = pd.read_csv("aqi_india_38cols_knn_final.csv")

In [4]:
df.head()

,city,state,latitude,longitude,datetime,month,day_name,is_weekend,season,time_of_day,...,no2_ugm3,so2_ugm3,o3_ugm3,dust_ugm3,aod,us_aqi,aqi_category,pm25_category_india,festival_period,crop_burning_season
0,agartala,tripura,23.8315,91.2868,2022-08-05 00:00:00,8.0,friday,False,monsoon,night,...,21.8,2.7,32.0,0.0,0.14,289.0,Very Unhealthy,good,False,False
1,agartala,tripura,23.8315,91.2868,2022-08-05 01:00:00,8.0,friday,False,monsoon,night,...,18.5,3.0,33.0,0.0,0.14,289.0,Very Unhealthy,good,False,False
2,agartala,tripura,23.8315,91.2868,2022-08-05 02:00:00,8.0,friday,False,monsoon,night,...,15.1,3.3,34.0,0.0,0.15,54.0,Moderate,good,False,False
3,agartala,tripura,23.8315,91.2868,2022-08-05 03:00:00,8.0,friday,False,monsoon,night,...,14.1,3.3,32.0,0.0,0.15,54.0,Moderate,good,False,False
4,agartala,tripura,23.8315,91.2868,2022-08-05 04:00:00,8.0,friday,False,monsoon,night,...,13.9,3.2,30.0,0.0,0.14,54.0,Moderate,good,False,False


In [16]:
df.sample(5)


,city,state,latitude,longitude,datetime,month,day_name,is_weekend,season,time_of_day,...,no2_ugm3,so2_ugm3,o3_ugm3,dust_ugm3,aod,us_aqi,aqi_category,pm25_category_india,festival_period,crop_burning_season
364448,guwahati,assam,26.1445,91.7362,2024-05-31 08:00:00,5.0,friday,False,summer,early_morning,...,1.8,1.9,93.0,14.0,1.07,71.0,Moderate,good,False,False
704470,ranchi,jharkhand,23.3441,85.3096,2023-06-13 22:00:00,6.0,tuesday,False,monsoon,night_late,...,14.7,7.0,84.0,11.0,0.67,85.0,Moderate,good,False,False
745249,shillong,meghalaya,25.5788,91.8933,2024-10-15 01:00:00,10.0,tuesday,False,post_monsoon,night,...,5.1,0.7,28.0,1.0,0.78,90.0,Moderate,good,True,True
803936,thiruvananthapuram,kerala,8.5241,76.9366,2024-11-09 08:00:00,11.0,saturday,True,post_monsoon,early_morning,...,4.1,5.1,95.0,0.0,0.29,39.0,Good,good,True,True
196769,chandigarh,punjab,30.7333,76.7794,2025-02-28 17:00:00,2.0,friday,False,winter,afternoon,...,11.4,3.8,84.0,41.0,0.57,65.0,Moderate,good,False,False


In [5]:
print("Rows :", df.shape[0])
print("Columns :", df.shape[1])

Rows : 842160
Columns : 31


In [7]:
df.columns

Index(['city', 'state', 'latitude', 'longitude', 'datetime', 'month',
       'day_name', 'is_weekend', 'season', 'time_of_day', 'humidity_percent',
       'dew_point_c', 'wind_gusts_kmh', 'precipitation_mm', 'is_raining',
       'heavy_rain', 'pressure_msl_hpa', 'cloud_cover_percent', 'pm2_5_ugm3',
       'pm10_ugm3', 'co_ugm3', 'no2_ugm3', 'so2_ugm3', 'o3_ugm3', 'dust_ugm3',
       'aod', 'us_aqi', 'aqi_category', 'pm25_category_india',
       'festival_period', 'crop_burning_season'],
      dtype='object')

In [6]:
df.dtypes

,0
city,object
state,object
latitude,float64
longitude,float64
datetime,object
month,float64
day_name,object
is_weekend,bool
season,object
time_of_day,object


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 842160 entries, 0 to 842159
Data columns (total 31 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   city                 842160 non-null  object 
 1   state                842160 non-null  object 
 2   latitude             842160 non-null  float64
 3   longitude            842160 non-null  float64
 4   datetime             842160 non-null  object 
 5   month                842160 non-null  float64
 6   day_name             842160 non-null  object 
 7   is_weekend           842160 non-null  bool   
 8   season               842160 non-null  object 
 9   time_of_day          842160 non-null  object 
 10  humidity_percent     842160 non-null  float64
 11  dew_point_c          842160 non-null  float64
 12  wind_gusts_kmh       842160 non-null  float64
 13  precipitation_mm     842160 non-null  float64
 14  is_raining           842160 non-null  bool   
 15  heavy_rain       

In [20]:
df.describe()

,latitude,longitude,datetime,month,humidity_percent,dew_point_c,wind_gusts_kmh,precipitation_mm,pressure_msl_hpa,cloud_cover_percent,pm2_5_ugm3,pm10_ugm3,co_ugm3,no2_ugm3,so2_ugm3,o3_ugm3,dust_ugm3,aod,us_aqi
count,842160.000000,842160.000000,842160,842160.000000,842160.000000,842160.000000,842160.000000,842160.000000,842160.000000,842160.00000,842160.000000,842160.000000,842160.000000,842160.000000,842160.000000,842160.000000,842160.000000,842160.000000,842160.000000
mean,23.129041,82.769207,2024-03-31 23:30:00,6.801653,71.324796,17.160777,19.270394,0.197799,1009.489776,50.50123,34.549825,54.014723,444.932728,15.991785,13.390032,80.188893,19.856400,0.463936,94.712182
min,8.524100,72.571400,2022-08-05 00:00:00,1.000000,3.000000,-11.500000,3.200000,0.000000,979.400000,0.00000,1.900000,2.300000,96.000000,0.000000,0.200000,0.000000,0.000000,0.050000,19.000000
25%,20.296100,77.173400,2023-06-03 11:45:00,4.000000,58.000000,12.100000,11.500000,0.000000,1005.400000,2.00000,14.700000,21.500000,235.000000,3.700000,2.800000,47.000000,0.000000,0.240000,59.000000
50%,23.831500,80.946200,2024-03-31 23:30:00,7.000000,76.000000,18.600000,17.300000,0.000000,1009.900000,47.00000,25.900000,39.000000,333.000000,9.000000,6.900000,73.000000,2.000000,0.400000,84.000000
75%,26.912400,88.606500,2025-01-28 11:15:00,10.000000,89.000000,23.300000,25.200000,0.000000,1013.800000,100.00000,44.200000,66.400000,508.000000,19.900000,15.800000,108.000000,11.000000,0.610000,128.000000
max,31.104800,94.108600,2025-11-26 23:00:00,12.000000,100.000000,27.300000,55.100000,51.700000,1028.100000,100.00000,183.500000,420.000000,2903.000000,122.100000,134.200000,228.000000,714.000000,1.700000,289.000000
std,5.543478,6.937205,NaN,3.411504,21.362100,7.020427,9.921461,0.915802,5.698795,43.41348,29.652013,54.231554,377.007222,19.896299,18.921014,44.762719,72.359802,0.299889,47.234756


In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df["city"].nunique()

29

In [11]:
df["state"].nunique()

29

In [12]:
sorted(df["city"].unique())

['agartala',
 'ahmedabad',
 'aizawl',
 'bengaluru',
 'bhopal',
 'bhubaneswar',
 'chandigarh',
 'chennai',
 'dehradun',
 'delhi',
 'gangtok',
 'gurugram',
 'guwahati',
 'hyderabad',
 'imphal',
 'itanagar',
 'jaipur',
 'kohima',
 'kolkata',
 'lucknow',
 'mumbai',
 'panaji',
 'patna',
 'raipur',
 'ranchi',
 'shillong',
 'shimla',
 'thiruvananthapuram',
 'visakhapatnam']

In [13]:
sorted(df["state"].unique())

['andhra pradesh',
 'arunachal pradesh',
 'assam',
 'bihar',
 'chhattisgarh',
 'delhi',
 'goa',
 'gujarat',
 'haryana',
 'himachal pradesh',
 'jharkhand',
 'karnataka',
 'kerala',
 'madhya pradesh',
 'maharashtra',
 'manipur',
 'meghalaya',
 'mizoram',
 'nagaland',
 'odisha',
 'punjab',
 'rajasthan',
 'sikkim',
 'tamil nadu',
 'telangana',
 'tripura',
 'uttar pradesh',
 'uttarakhand',
 'west bengal']

In [14]:
df["datetime"] = pd.to_datetime(df["datetime"])

print(df["datetime"].min())
print(df["datetime"].max())

2022-08-05 00:00:00
2025-11-26 23:00:00


In [15]:
df["us_aqi"].describe()

,us_aqi
count,842160.000000
mean,94.712182
std,47.234756
min,19.000000
25%,59.000000
50%,84.000000
75%,128.000000
max,289.000000


In [17]:
missing = df.isnull().sum()

missing[missing > 0]

,0
pm25_category_india,2


In [19]:
df.describe(include="object")

,city,state,day_name,season,time_of_day,aqi_category,pm25_category_india
count,842160,842160,842160,842160,842160,842160,842158
unique,29,29,7,4,6,5,6
top,agartala,tripura,friday,monsoon,night,Moderate,good
freq,29040,29040,120408,294408,210540,385833,482562


In [22]:
df.select_dtypes(include="object").columns

Index(['city', 'state', 'day_name', 'season', 'time_of_day', 'aqi_category',
       'pm25_category_india'],
      dtype='object')

In [21]:
df.select_dtypes(include="bool").columns

Index(['is_weekend', 'is_raining', 'heavy_rain', 'festival_period',
       'crop_burning_season'],
      dtype='object')